# 08 — Experimento 3: explicación de las alertas con el modelo de lenguaje

¿Explica el modelo de lenguaje cada alerta de forma útil, legible y fiel a los datos? (apartado 5.3 de la memoria). Este notebook prepara y ejecuta el experimento; la evaluación y H₃ están en `09_experimento_3_resultados`.

**Diseño:** Qwen3.5-9B (4 bits, sin muestreo, razonamiento desactivado) recibe la ventana como tabla de señales en filas con **tres filas más calculadas en Python**: lo que queda sin recuperar, el porcentaje de la pérdida en ráfaga y el porcentaje recuperado. En las pruebas de desarrollo, cuando el modelo tenía que hacer esas cuentas, se equivocaba y el error arrastraba el diagnóstico; por eso se le dan hechas (Russell-Gilbert et al., 2024). El prompt responde 9 preguntas antes de concluir y devuelve un JSON con la cadena de causas, la evolución, el origen, el impacto en vídeo y en audio y una explicación para el operador.

**Escala de impacto (supuesto declarado):** vídeo y audio por separado, cada uno con su peor reporte: ninguno = 0; bajo = 1 a 10 (p90 de los valores distintos de cero en entrenamiento); medio = 11 a 360; alto = más de 360 (Begen, 2009).

| Sección | Qué hace | Entorno |
|---|---|---|
| 1 | Entorno y comprobación del prompt (misma huella que en la memoria) | CPU |
| 2 | Ventanas: las 20 alertas de consenso, el resto de prueba y las de desarrollo | CPU |
| 3 | Copia, comprobada, de las ejecuciones ya hechas | CPU |
| 4 | Prueba en una ventana de desarrollo (opcional) | GPU |
| 5 | Ejecución sobre un conjunto de ventanas (solo para repetir o completar) | GPU (L4 recomendada) |

**Prompt:** vive en `privado/prompts` (variante `exp3_calculos`), con los nombres cortos; `llm_lab.prompts` pone los nombres reales al montar cada mensaje.

**Avisos metodológicos:** el prompt se ajustó mirando ventanas de desarrollo; las puntúa un solo evaluador.

**Confidencial:** este notebook solo muestra cifras y nombres cortos. Las respuestas del modelo se guardan en `privado/experimento_3/` y se leen en el informe HTML del `09_`. No subas al repositorio versiones con otras salidas.

## 1. Entorno

En Colab, monta Google Drive y clona el repositorio para tener acceso a `src/`. En local, busca la carpeta `src/` subiendo desde el directorio actual.

In [ ]:
# --- Entorno: localiza src/ y, en Colab, monta Google Drive ---
import os
import subprocess
import sys

EN_COLAB = "google.colab" in sys.modules
URL_REPOSITORIO = "https://github.com/Williams-MP/TFM_VIU_2026.git"
RUTA_REPO_COLAB = "/content/TFM_VIU_2026"

if EN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    if not os.path.isdir(RUTA_REPO_COLAB):
        # Mientras el repositorio sea privado hace falta un token de GitHub.
        # Si es público, basta con dejar el campo vacío.
        from getpass import getpass
        token = getpass("Token de GitHub (vacío si el repositorio es público): ")
        url = URL_REPOSITORIO.replace("https://", f"https://{token}@") if token else URL_REPOSITORIO
        subprocess.run(["git", "clone", "-q", url, RUTA_REPO_COLAB], check=True)
    ruta_src = f"{RUTA_REPO_COLAB}/src"
else:
    # Ejecución local: se sube por las carpetas hasta encontrar src/.
    directorio = os.getcwd()
    while not os.path.isdir(os.path.join(directorio, "src")):
        padre = os.path.dirname(directorio)
        if padre == directorio:
            raise FileNotFoundError("No se encuentra la carpeta src/ del repositorio.")
        directorio = padre
    ruta_src = os.path.join(directorio, "src")

if ruta_src not in sys.path:
    sys.path.append(ruta_src)

import glob
import json

import pandas as pd

from config import RUTA_PREPARADOS, IDX_CONTADORES, IDX_GAP, RUTA_EXPERIMENTO_2_PRIVADO, RUTA_EXPERIMENTO_3_PRIVADO
from llm_lab import exp2, exp3
from llm_lab.huellas import huella_texto
from llm_lab.prompts import Prompts
from llm_lab.ventanas import FuenteVentanas

RUTA_ENTRADAS = f"{RUTA_EXPERIMENTO_2_PRIVADO}/entradas"            # las mismas ventanas y referencia del Experimento 2
RUTA_EJECUCIONES = f"{RUTA_EXPERIMENTO_3_PRIVADO}/ejecuciones"      # prompt exacto y respuestas (privado)
RUTA_PRUEBAS = f"{RUTA_EXPERIMENTO_3_PRIVADO}/pruebas"              # pruebas en una ventana (privado)

prompts = Prompts()                                                  # lee privado/prompts y nombres_senales.json
fuente = FuenteVentanas(RUTA_PREPARADOS, IDX_CONTADORES, IDX_GAP)

In [ ]:
# --- 1.2 Variante del prompt y su huella ---
VARIANTE = "exp3_calculos@v1"
HUELLA_MEMORIA = "30ff1bead2e8"   # huella del prompt con el que se hicieron las ejecuciones de la memoria

r = prompts.resolver(VARIANTE)
HUELLA = huella_texto(prompts.texto_prompt(VARIANTE))
print(f"{r['id']} · representación: {r['representaciones']} · orden: {' -> '.join(r['orden'])} · "
      f"tokens máx.: {r['max_new_tokens']}")
print(f"Huella del prompt: {HUELLA}", "(la de la memoria)" if HUELLA == HUELLA_MEMORIA else "(DISTINTA de la memoria)")

## 2. Ventanas

Las mismas del Experimento 2 (`privado/experimento_2/entradas`, creadas en el `05_`), con la huella de cada una comprobada contra los datos preparados:
- **Alertas de consenso (20):** las ventanas de prueba que TranAD y TranAD+ marcan a la vez. Son las que cuentan para H₃.
- **Resto de prueba (20):** 10 de alta actividad sin alerta y 10 al azar; sirven para ver cómo explica el modelo las ventanas normales y mitigadas.
- **Desarrollo (29):** las que se usaron para ajustar el prompt.

In [ ]:
# --- 2. Ventanas: alertas de consenso, resto de prueba y desarrollo ---
prueba = exp2.cargar_ventanas(f"{RUTA_ENTRADAS}/ventanas_prueba.csv", fuente)
desarrollo = exp2.cargar_ventanas(f"{RUTA_ENTRADAS}/ventanas_desarrollo.csv", fuente)

ALERTAS = sorted([w for w in prueba if w["grupo"] == "alerta_consenso_comun"], key=lambda w: w["id"])
RESTO_PRUEBA = sorted([w for w in prueba if w["grupo"] != "alerta_consenso_comun"], key=lambda w: w["id"])
TODAS = ALERTAS + RESTO_PRUEBA + desarrollo
CONJUNTOS = {"alertas": ALERTAS, "prueba": ALERTAS + RESTO_PRUEBA, "desarrollo": desarrollo, "todas": TODAS}
print(f"Alertas de consenso: {len(ALERTAS)} · resto de prueba: {len(RESTO_PRUEBA)} · desarrollo: {len(desarrollo)}")
display(pd.crosstab(pd.Series([w["grupo"] for w in TODAS], name="grupo"),
                    pd.Series([w["referencia"] for w in TODAS], name="referencia"), margins=True, margins_name="total"))

## 3. Ejecuciones ya hechas

**(Una sola vez, solo el autor.)** Copia a `privado/experimento_3/ejecuciones/` las ejecuciones del repositorio de trabajo hechas con este mismo prompt. Antes de copiar cada una se comprueba que tiene la huella de la 1.2 y que, ventana a ventana, el modelo recibió **exactamente los mismos mensajes** que se montan ahora; si una sola ventana no coincide, no se copia. Las ejecuciones con otra huella (versiones anteriores del prompt) se omiten. La carpeta de origen solo se lee.

In [ ]:
# --- 3. Copia comprobada de las ejecuciones hechas con este prompt ---
ORIGEN_EXP3 = "/content/drive/MyDrive/TFM_VIU/export_colab_split_estratificado/experimento_3/ejecuciones"   # solo lectura
PREFIJO = "exp3_calc"

filas = []
for ruta in sorted(glob.glob(f"{ORIGEN_EXP3}/{PREFIJO}_*.json")):
    nombre = os.path.basename(ruta)
    huella = json.load(open(ruta, encoding="utf-8")).get("huella_prompt")
    if huella != HUELLA:
        filas.append({"ejecucion": nombre, "huella": huella, "resultado": "omitida (otro prompt)"})
    elif os.path.exists(f"{RUTA_EJECUCIONES}/{nombre}") and os.path.getmtime(f"{RUTA_EJECUCIONES}/{nombre}") >= os.path.getmtime(ruta):
        filas.append({"ejecucion": nombre, "huella": huella, "resultado": "ya copiada"})
    else:
        exp3.copiar_ejecucion(ruta, RUTA_EJECUCIONES, prompts, VARIANTE, fuente, TODAS)
        filas.append({"ejecucion": nombre, "huella": huella, "resultado": "copiada"})
display(pd.DataFrame(filas))

# Ejecuciones disponibles: cuántas ventanas de cada conjunto tiene cada una
disponibles = []
for ruta in sorted(glob.glob(f"{RUTA_EJECUCIONES}/{PREFIJO}_*.json")):
    ids = {c["id"] for c in json.load(open(ruta, encoding="utf-8"))["llamadas"]}
    disponibles.append({"ejecucion": os.path.basename(ruta)[:-5],
                        **{c: f"{len(ids & {w['id'] for w in v})} de {len(v)}" for c, v in CONJUNTOS.items() if c != "todas"}})
display(pd.DataFrame(disponibles))

## 4. Prueba en una ventana (opcional, GPU)

Solo con ventanas de **desarrollo** (`val_…`). Muestra el número de citas y de valores calculados correctos, la evolución y el impacto; la respuesta completa se guarda en `privado/experimento_3/pruebas/pruebas.jsonl`.

In [ ]:
# --- 4.1 Modelo (GPU): se carga una vez para la 4.2 y la 5 ---
CARGAR_MODELO = False     # True para probar (4.2) o ejecutar (5)
MODELO = "qwen3.5-9b"     # clave de llm_lab.modelos.CATALOGO_MODELOS

modelo = None
if CARGAR_MODELO:
    import importlib.util
    if importlib.util.find_spec("bitsandbytes") is None:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "bitsandbytes"])
        raise SystemExit("bitsandbytes instalado: reinicia la sesión y vuelve a ejecutar desde la sección 1")
    from llm_lab.modelos import ModeloLLM
    modelo = ModeloLLM(MODELO)
    print("Modelo cargado:", modelo.parametros)
else:
    print("Sin modelo (CARGAR_MODELO = False): las secciones 4.2 y 5 no ejecutan nada")

In [ ]:
# --- 4.2 Probar en una ventana de desarrollo ---
ID_VENTANA = "val_9971"

if modelo is not None:
    w = next(w for w in desarrollo if w["id"] == ID_VENTANA)   # solo desarrollo
    v = fuente.ventana(w["conjunto"], w["indice"], avisar=False)
    analisis = exp3.probar(prompts, VARIANTE, modelo, v, w, registro=f"{RUTA_PRUEBAS}/pruebas.jsonl")

## 5. Ejecución (GPU; solo para repetir o completar)

Las ejecuciones de la memoria ya están copiadas en la sección 3. Esta celda solo hace falta para repetir una o para completar un conjunto. Con una L4, unos 100 s por ventana: 20 alertas ≈ 35 min, 40 de prueba ≈ 70 min, 29 de desarrollo ≈ 50 min. Se guarda tras cada ventana; si Colab se corta, pon en `REANUDAR` el nombre de la ejecución y vuelve a lanzarla.

In [ ]:
# --- 5. Ejecución: solo si EJECUTAR = True (y el modelo cargado en la 4.1) ---
CONJUNTO_EJECUCION = "prueba"   # "alertas" (20) | "prueba" (40) | "desarrollo" (29) | "todas" (69)
EJECUTAR = False
CONFIRMO_PRUEBA = False         # obligatorio en True si hay ventanas de prueba
REANUDAR = None                 # nombre de una ejecución interrumpida, p. ej. "exp3_calc_qwen3.5-9b_20261013_190000"

if EJECUTAR and modelo is not None:
    RUTA_EJECUCION = exp3.ejecutar(prompts, VARIANTE, modelo, fuente, CONJUNTOS[CONJUNTO_EJECUCION], RUTA_EJECUCIONES,
                                   prefijo=PREFIJO, confirmo_prueba=CONFIRMO_PRUEBA, reanudar=REANUDAR)
else:
    print("No se ejecuta nada (EJECUTAR = False o sin modelo)")